In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Flipkart Data Pipeline").getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

# Write to Bronze (RAW)
df.write.format("delta").mode("append").save("/Volumes/workspace/default/practice/flipkart/bronze")

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/practice/flipkart/bronze")

In [0]:
from pyspark.sql.functions import col, to_date
df = df.withColumn("amount", col("amount").cast("int"))\
       .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

In [0]:
df = df.fillna({
    "amount": "0",
    "city": "Unknown"
})

In [0]:
df = df.filter(col("amount") >= 0)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("order_id").orderBy(col("date").desc())

df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1) \
       .drop("rn")

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df = df.dropDuplicates()

In [0]:
df.write.format("delta").mode("append").save("/Volumes/workspace/default/practice/flipkart/silver")

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/practice/flipkart/silver")

In [0]:
from pyspark.sql.functions import sum
sales_by_product = df.groupBy("product").agg(sum(col("amount")).alias("total_sales_by_product"))

In [0]:
sales_by_product.display()

product,total_sales_by_product
Tablet,220000
Mobile,330000
Watch,80000
Laptop,1050000
Headphones,30000


In [0]:
from pyspark.sql.functions import sum
sales_by_category = df.groupBy("category").agg(sum(col("amount")).alias("total_sales_by_category"))

In [0]:
sales_by_category.display()

category,total_sales_by_category
Electronics,1600000
Accessories,110000


In [0]:
sales_by_city = df.groupBy("city").agg(sum(col("amount")).alias("total_sales_by_city"))

In [0]:
sales_by_city.display()

city,total_sales_by_city
Delhi,550000
Chennai,330000
Unknown,30000
Hyderabad,720000
Mumbai,80000


In [0]:
orders_per_customer = df.groupBy("customer_id").count().withColumnRenamed("count", "total_orders")

In [0]:
orders_per_customer.display()

customer_id,total_orders
C005,10
C004,10
C003,10
C001,20
C002,20
C007,10


In [0]:
customer_spending = df.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spent"))

In [0]:
customer_spending.display()

customer_id,total_spent
C005,30000
C004,80000
C003,550000
C001,720000
C002,180000
C007,150000


In [0]:
top_product = sales_by_product.orderBy(col("total_sales").desc()).limit(1)

In [0]:
top_customer = customer_spending.orderBy(col("total_spent").desc()).limit(1)

In [0]:
top_customer.display()

customer_id,total_spent
C001,720000


In [0]:
sales_by_product.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/practice/flipkart/gold/sales_by_product")

sales_by_category.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/practice/flipkart/gold/sales_by_category")

sales_by_city.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/practice/flipkart/gold/sales_by_city")

orders_per_customer.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/practice/flipkart/gold/orders_per_customer")

customer_spending.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/default/practice/flipkart/gold/customer_spending")